> **Repository version.** This notebook was cleaned from the frozen manuscript workflow:
> outputs were removed and local absolute paths were replaced by the repository `PROJECT_DIR`.
> The statistical logic was not intentionally changed.


In [ ]:
# Repository path setup
from pathlib import Path
import os

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
PROJECT_DIR = Path(
    os.environ.get("CGN_PROJECT_DIR", REPO_ROOT / "workspace")
).expanduser().resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "reproduction" / "results").mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Working data/results directory:", PROJECT_DIR)


# FINAL MANUSCRIPT FIGURES 1–4

Final terminology-synchronised submission figures.

## Figure 1

In [ ]:
# ============================================================
# FINAL MANUSCRIPT FIGURE 1 — REVISED
#
# Analytical framework for progression-matched
# cross-etiology analysis
#
# A  Study design
# B  Published study cohort
# C  Crescent PC1 agrees with diffusion pseudotime
# D  Crescent-axis-matched analytical framework
#
# IMPORTANT:
# Study cohort = 63 patient samples:
#   Control = 6
#   LN = 19
#   ANCA = 32
#   anti-GBM = 6
#
# ROI-analysis availability may be smaller.
# Do NOT confuse analytical subset with the study cohort.
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path

import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch
from scipy.stats import spearmanr


# ============================================================
# 1. Paths
# ============================================================

base = Path(
    str(PROJECT_DIR)
)

figdir = (
    base /
    "figures"
)

figdir.mkdir(
    parents=True,
    exist_ok=True
)


roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)

big_path = (
    base /
    "GSE294965_processed_data.h5ad"
)


if not roi_path.exists():

    raise FileNotFoundError(
        f"找不到：{roi_path}"
    )


# ============================================================
# 2. Load ROI object
# ============================================================

roi_ad = ad.read_h5ad(
    roi_path
)


meta = (
    roi_ad.obs.copy()
)

meta.index = (
    meta.index.astype(str)
)


print(
    "ROI object:",
    roi_ad.shape
)


# ============================================================
# 3. Standardize disease names
# ============================================================

def standardize_disease(x):

    x = str(x)

    mapping = {

        "Cntrl":
            "Control",

        "Ctrl":
            "Control",

        "Control":
            "Control",

        "SLE":
            "LN",

        "LN":
            "LN",

        "ANCA":
            "ANCA",

        "GBM":
            "anti-GBM",

        "anti-GBM":
            "anti-GBM"
    }

    return mapping.get(
        x,
        x
    )


meta[
    "Disease_display"
] = (
    meta[
        "Disease"
    ]
    .map(
        standardize_disease
    )
)


# ============================================================
# 4. FIXED STUDY COHORT
#
# These are the full published study-cohort counts.
#
# Do NOT derive these from the ROI analysis object,
# because some samples may not contribute usable ROIs.
# ============================================================

cohort_order = [
    "Control",
    "LN",
    "ANCA",
    "anti-GBM"
]


study_cohort_counts = pd.Series(
    {
        "Control":
            6,

        "LN":
            19,

        "ANCA":
            32,

        "anti-GBM":
            6
    }
).reindex(
    cohort_order
)


study_total = int(
    study_cohort_counts.sum()
)


print(
    "\n================================"
)

print(
    "FULL STUDY COHORT"
)

print(
    "================================"
)


print(
    study_cohort_counts
)


print(
    "\nTotal =",
    study_total
)


# ============================================================
# 5. Separately audit ROI-analysis patient availability
#
# This is NOT used for Panel B.
# ============================================================

if "Patient_Sample_ID" in meta.columns:

    analysis_patient_counts = (
        meta[
            [
                "Disease_display",
                "Patient_Sample_ID"
            ]
        ]
        .drop_duplicates()
        .groupby(
            "Disease_display",
            observed=True
        )[
            "Patient_Sample_ID"
        ]
        .nunique()
        .reindex(
            cohort_order
        )
        .fillna(
            0
        )
        .astype(
            int
        )
    )


    print(
        "\n================================"
    )

    print(
        "ROI-ANALYSIS PATIENT AVAILABILITY"
    )

    print(
        "================================"
    )


    print(
        analysis_patient_counts
    )


    print(
        "\nROI-analysis total =",
        int(
            analysis_patient_counts.sum()
        )
    )


else:

    analysis_patient_counts = None


# ============================================================
# 6. Dataset dimensions
#
# Read large h5ad in backed mode only.
# ============================================================

n_cells_total = (
    3_218_210
)

n_genes_total = (
    480
)


if big_path.exists():

    try:

        big_backed = ad.read_h5ad(
            big_path,
            backed="r"
        )


        n_cells_total = (
            big_backed.n_obs
        )


        n_genes_total = (
            big_backed.n_vars
        )


        try:

            big_backed.file.close()

        except Exception:

            pass


    except Exception as e:

        print(
            "\nLarge h5ad backed read failed."
        )

        print(
            "Using known dimensions:"
        )

        print(
            n_cells_total,
            "cells ×",
            n_genes_total,
            "genes"
        )


# ============================================================
# 7. Obtain diffusion pseudotime
#
# Prefer saved DPT column.
# Otherwise recompute on small 782 × 480 ROI object.
# ============================================================

possible_dpt_columns = [
    "dpt_pseudotime",
    "DPT",
    "dpt",
    "diffusion_pseudotime",
    "pseudotime"
]


dpt_column = None


for c in possible_dpt_columns:

    if c in meta.columns:

        dpt_column = c

        break


if dpt_column is not None:

    print(
        "\nUsing existing DPT column:",
        dpt_column
    )


    dpt = (
        pd.to_numeric(
            meta[
                dpt_column
            ],
            errors="coerce"
        )
        .to_numpy()
    )


else:

    print(
        "\nNo stored DPT found."
    )

    print(
        "Recomputing diffusion pseudotime "
        "on the 782-ROI object..."
    )


    dpt_ad = (
        roi_ad.copy()
    )


    if "X_pca" not in dpt_ad.obsm:

        sc.pp.pca(
            dpt_ad,
            n_comps=min(
                30,
                dpt_ad.n_vars - 1
            )
        )


    sc.pp.neighbors(
        dpt_ad,
        n_neighbors=15,
        use_rep="X_pca"
    )


    sc.tl.diffmap(
        dpt_ad
    )


    if "PC1_crescent" not in dpt_ad.obs.columns:

        raise ValueError(
            "没有找到 PC1_crescent。"
        )


    pc1_tmp = (
        pd.to_numeric(
            dpt_ad.obs[
                "PC1_crescent"
            ],
            errors="coerce"
        )
        .to_numpy()
    )


    root_index = int(
        np.nanargmin(
            pc1_tmp
        )
    )


    dpt_ad.uns[
        "iroot"
    ] = root_index


    sc.tl.dpt(
        dpt_ad
    )


    dpt = (
        dpt_ad.obs[
            "dpt_pseudotime"
        ]
        .to_numpy(
            dtype=float
        )
    )


# ============================================================
# 8. PC1–DPT agreement
# ============================================================

pc1 = (
    pd.to_numeric(
        meta[
            "PC1_crescent"
        ],
        errors="coerce"
    )
    .to_numpy()
)


valid = (
    np.isfinite(
        pc1
    )
    &
    np.isfinite(
        dpt
    )
)


rho, rho_p = (
    spearmanr(
        pc1[
            valid
        ],
        dpt[
            valid
        ]
    )
)


print(
    "\n================================"
)

print(
    "PC1–DPT AGREEMENT"
)

print(
    "================================"
)


print(
    "Spearman rho =",
    rho
)


print(
    "P =",
    rho_p
)


# ============================================================
# 9. Three-disease common PC1 support
# ============================================================

disease_meta = (
    meta[
        meta[
            "Disease"
        ].isin(
            [
                "ANCA",
                "SLE",
                "GBM"
            ]
        )
    ]
    .copy()
)


ranges = (
    disease_meta
    .groupby(
        "Disease",
        observed=True
    )[
        "PC1_crescent"
    ]
    .agg(
        [
            "min",
            "max"
        ]
    )
)


pc_low = float(
    ranges[
        "min"
    ].max()
)


pc_high = float(
    ranges[
        "max"
    ].min()
)


print(
    "\nCommon PC1 support:"
)

print(
    f"{pc_low:.2f}",
    "to",
    f"{pc_high:.2f}"
)


# ============================================================
# 10. Figure typography
# ============================================================

plt.rcParams[
    "pdf.fonttype"
] = 42

plt.rcParams[
    "ps.fonttype"
] = 42

plt.rcParams[
    "font.size"
] = 9.5

plt.rcParams[
    "axes.titlesize"
] = 11

plt.rcParams[
    "axes.labelsize"
] = 10

plt.rcParams[
    "legend.fontsize"
] = 8


# ============================================================
# 11. Colors
# ============================================================

disease_colors = {

    "Control":
        "#A6A6A6",

    "LN":
        "#E69F00",

    "ANCA":
        "#56B4E9",

    "anti-GBM":
        "#009E73"
}


# ============================================================
# 12. Figure canvas
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        13.5,
        10
    ),
    constrained_layout=True
)


axA = axes[
    0,
    0
]

axB = axes[
    0,
    1
]

axC = axes[
    1,
    0
]

axD = axes[
    1,
    1
]


# ============================================================
# 13. Helper: rounded box
# ============================================================

def add_box(
    ax,
    xy,
    width,
    height,
    text,
    fontsize=10,
    linewidth=1.2
):

    x, y = xy


    patch = FancyBboxPatch(
        (
            x,
            y
        ),
        width,
        height,
        boxstyle="round,pad=0.02",
        linewidth=linewidth,
        edgecolor="black",
        facecolor="white"
    )


    ax.add_patch(
        patch
    )


    ax.text(
        x
        +
        width / 2,
        y
        +
        height / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize
    )


# ============================================================
# 14. PANEL A — Study design
# ============================================================

axA.set_xlim(
    0,
    1
)

axA.set_ylim(
    0,
    1
)

axA.axis(
    "off"
)


axA.set_title(
    "A  Study design",
    loc="left",
    fontweight="normal"
)


add_box(
    axA,
    (
        0.12,
        0.73
    ),
    0.76,
    0.14,
    (
        "Human kidney biopsies\n"
        "63 patient samples"
    )
)


axA.annotate(
    "",
    xy=(
        0.50,
        0.68
    ),
    xytext=(
        0.50,
        0.73
    ),
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.2
    )
)


add_box(
    axA,
    (
        0.12,
        0.49
    ),
    0.76,
    0.14,
    (
        "Xenium spatial transcriptomics\n"
        f"{n_cells_total:,} cells × "
        f"{n_genes_total} genes"
    )
)


axA.annotate(
    "",
    xy=(
        0.50,
        0.44
    ),
    xytext=(
        0.50,
        0.49
    ),
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.2
    )
)


add_box(
    axA,
    (
        0.12,
        0.25
    ),
    0.76,
    0.14,
    (
        "782 glomerular/periglomerular ROIs\n"
        "cell composition + molecular states + "
        "spatial neighborhoods"
    )
)


axA.text(
    0.50,
    0.10,
    "Crescent-axis-matched cross-etiology analysis",
    ha="center",
    va="center",
    fontsize=10,
    fontweight="bold"
)


# ============================================================
# 15. PANEL B — Full study cohort
# ============================================================

x = np.arange(
    len(
        cohort_order
    )
)


values = (
    study_cohort_counts
    .reindex(
        cohort_order
    )
    .to_numpy()
)


bars = axB.bar(
    x,
    values,
    color=[
        disease_colors[
            d
        ]
        for d in cohort_order
    ],
    edgecolor="none"
)


axB.set_xticks(
    x
)


axB.set_xticklabels(
    cohort_order,
    rotation=20,
    ha="right"
)


axB.set_ylabel(
    "Number of patients"
)


axB.set_title(
    "B  Study cohort",
    loc="left",
    fontweight="normal"
)


for bar, value in zip(
    bars,
    values
):

    axB.text(
        bar.get_x()
        +
        bar.get_width() / 2,
        bar.get_height()
        +
        0.4,
        str(
            int(
                value
            )
        ),
        ha="center",
        va="bottom",
        fontsize=9
    )


axB.set_ylim(
    0,
    max(
        values
    )
    *
    1.15
)


axB.spines[
    "top"
].set_visible(
    False
)


axB.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# 16. PANEL C — PC1 vs diffusion pseudotime
# ============================================================

for disease in cohort_order:

    mask = (
        meta[
            "Disease_display"
        ].to_numpy()
        ==
        disease
    )


    mask = (
        mask
        &
        valid
    )


    axC.scatter(
        pc1[
            mask
        ],
        dpt[
            mask
        ],
        s=18,
        alpha=0.45,
        color=disease_colors[
            disease
        ],
        label=disease
    )


axC.set_xlabel(
    "Crescent-associated PC1"
)


axC.set_ylabel(
    "Diffusion pseudotime"
)


axC.set_title(
    "C  PC1 agrees with diffusion pseudotime",
    loc="left",
    fontweight="normal"
)


axC.text(
    0.04,
    0.94,
    (
        f"Spearman ρ = {rho:.3f}\n"
        f"P = {rho_p:.1e}"
    ),
    transform=axC.transAxes,
    ha="left",
    va="top",
    fontsize=9
)


axC.legend(
    frameon=False,
    loc="lower right"
)


axC.spines[
    "top"
].set_visible(
    False
)


axC.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# 17. PANEL D — Analytical workflow
# ============================================================

axD.set_xlim(
    0,
    1
)

axD.set_ylim(
    0,
    1
)

axD.axis(
    "off"
)


axD.set_title(
    "D  Crescent-axis-matched analytical framework",
    loc="left",
    fontweight="normal"
)


add_box(
    axD,
    (
        0.25,
        0.79
    ),
    0.50,
    0.10,
    "Shared crescent-associated PC1",
    fontsize=9.5
)


axD.annotate(
    "",
    xy=(
        0.50,
        0.70
    ),
    xytext=(
        0.50,
        0.79
    ),
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.2
    )
)


add_box(
    axD,
    (
        0.25,
        0.60
    ),
    0.50,
    0.10,
    (
        "Common PC1 support\n"
        f"{pc_low:.2f} to {pc_high:.2f}"
    ),
    fontsize=9
)


axD.annotate(
    "",
    xy=(
        0.50,
        0.51
    ),
    xytext=(
        0.50,
        0.60
    ),
    arrowprops=dict(
        arrowstyle="->",
        linewidth=1.2
    )
)


add_box(
    axD,
    (
        0.20,
        0.39
    ),
    0.60,
    0.12,
    (
        "Patient-balanced spline modeling\n"
        "Disease × PC1"
    ),
    fontsize=9.5
)


branch_y = (
    0.13
)

branch_width = (
    0.25
)

branch_height = (
    0.11
)


branch_x = [
    0.06,
    0.375,
    0.69
]


branch_text = [
    "Cell\ncomposition",
    "Molecular\nprograms",
    "Spatial\nneighborhoods"
]


for x0, text in zip(
    branch_x,
    branch_text
):

    axD.annotate(
        "",
        xy=(
            x0
            +
            branch_width / 2,
            branch_y
            +
            branch_height
        ),
        xytext=(
            0.50,
            0.39
        ),
        arrowprops=dict(
            arrowstyle="->",
            linewidth=1.0
        )
    )


    add_box(
        axD,
        (
            x0,
            branch_y
        ),
        branch_width,
        branch_height,
        text,
        fontsize=9
    )


# ============================================================
# 18. Save Figure 1
# ============================================================

png_path = (
    figdir /
    "Figure1_FINAL_progression_matched_framework_REVISED.png"
)


pdf_path = (
    figdir /
    "Figure1_FINAL_progression_matched_framework_REVISED.pdf"
)


plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


print(
    "\n========================================"
)

print(
    "FINAL REVISED FIGURE 1 COMPLETE"
)

print(
    "========================================"
)


print(
    "\nStudy cohort:"
)

print(
    study_cohort_counts
)


print(
    "\nTotal =",
    study_total
)


print(
    "\nPNG:"
)

print(
    png_path
)


print(
    "\nPDF:"
)

print(
    pdf_path
)

## Figure 2

In [ ]:
# ============================================================
# FINAL MANUSCRIPT FIGURE 2 — REVISED
#
# Major cellular composition changes are predominantly
# associated with the shared crescent progression axis
#
# A  MAC
# B  FIB
# C  EC
# D  POD
# E  Shared PC1 effect vs Disease × PC1 interaction
#
# Heatmap:
# low significance  -> white
# high significance -> dark blue
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. Paths
# ============================================================

base = Path(
    str(PROJECT_DIR)
)

figdir = (
    base /
    "figures"
)

figdir.mkdir(
    parents=True,
    exist_ok=True
)


roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)

cell_meta_path = (
    base /
    "roi782_cell_metadata.pkl"
)


for path in [
    roi_path,
    cell_meta_path
]:

    if not path.exists():

        raise FileNotFoundError(
            f"找不到文件：{path}"
        )


# ============================================================
# 2. Load data
# ============================================================

roi_ad = ad.read_h5ad(
    roi_path
)


roi_meta = (
    roi_ad.obs.copy()
)

roi_meta.index = (
    roi_meta.index.astype(str)
)


cells = pd.read_pickle(
    cell_meta_path
)


print(
    "ROI object:",
    roi_ad.shape
)


print(
    "ROI-associated cells:",
    cells.shape
)


# ============================================================
# 3. Metadata cleanup
# ============================================================

cells[
    "roi_id"
] = (
    cells[
        "roi_id"
    ].astype(str)
)


cells[
    "celltype_l1"
] = (
    cells[
        "celltype_l1"
    ].astype(str)
)


# ============================================================
# 4. Three-disease common PC1 support
# ============================================================

disease_order = [
    "ANCA",
    "SLE",
    "GBM"
]


disease_meta = (
    roi_meta[
        roi_meta[
            "Disease"
        ].isin(
            disease_order
        )
    ]
    .copy()
)


ranges = (
    disease_meta
    .groupby(
        "Disease",
        observed=True
    )[
        "PC1_crescent"
    ]
    .agg(
        [
            "min",
            "max"
        ]
    )
)


pc_low = float(
    ranges[
        "min"
    ].max()
)


pc_high = float(
    ranges[
        "max"
    ].min()
)


analysis_meta = (
    disease_meta[
        disease_meta[
            "PC1_crescent"
        ].between(
            pc_low,
            pc_high
        )
    ]
    .copy()
)


analysis_meta.index = (
    analysis_meta.index.astype(str)
)


print(
    "\nCommon PC1 support:"
)

print(
    pc_low,
    "to",
    pc_high
)


print(
    "\nROI counts:"
)

print(
    analysis_meta[
        "Disease"
    ].value_counts()
)


print(
    "\nPatient counts:"
)

print(
    analysis_meta
    .groupby(
        "Disease",
        observed=True
    )[
        "Patient_Sample_ID"
    ]
    .nunique()
)


# ============================================================
# 5. Resolve broad cell-type labels
# ============================================================

available_labels = set(
    cells[
        "celltype_l1"
    ]
    .dropna()
    .astype(str)
    .unique()
)


alias_map = {

    "MAC": [
        "MAC",
        "Macrophage"
    ],

    "Mono": [
        "Mono",
        "MONO",
        "Monocyte"
    ],

    "B": [
        "B",
        "B cell",
        "Bcell"
    ],

    "T": [
        "T",
        "T cell",
        "Tcell"
    ],

    "FIB": [
        "FIB",
        "Fibroblast"
    ],

    "EC": [
        "EC",
        "Endothelial"
    ],

    "PEC": [
        "PEC"
    ],

    "POD": [
        "POD",
        "Podocyte"
    ]
}


resolved_labels = {}


for desired, aliases in (
    alias_map.items()
):

    matches = [
        x
        for x in aliases
        if x in available_labels
    ]


    if len(
        matches
    ) == 0:

        raise ValueError(
            f"\n没有找到 {desired}。\n"
            f"候选 aliases = {aliases}\n"
            f"Available labels = "
            f"{sorted(available_labels)}"
        )


    resolved_labels[
        desired
    ] = matches[
        0
    ]


print(
    "\nResolved broad cell types:"
)

print(
    resolved_labels
)


# ============================================================
# 6. Keep common-support ROIs
# ============================================================

analysis_roi_ids = set(
    analysis_meta.index
)


cells_use = (
    cells[
        cells[
            "roi_id"
        ].isin(
            analysis_roi_ids
        )
    ]
    .copy()
)


# ============================================================
# 7. Total cell count per ROI
# ============================================================

total_by_roi = (
    cells_use
    .groupby(
        "roi_id",
        observed=True
    )
    .size()
    .rename(
        "total_cells"
    )
)


# ============================================================
# 8. ROI × broad-celltype counts
# ============================================================

celltype_order = [
    "MAC",
    "Mono",
    "B",
    "T",
    "FIB",
    "EC",
    "PEC",
    "POD"
]


count_df = pd.DataFrame(
    index=analysis_meta.index
)


for desired in celltype_order:

    exact_label = (
        resolved_labels[
            desired
        ]
    )


    counts = (
        cells_use[
            cells_use[
                "celltype_l1"
            ]
            ==
            exact_label
        ]
        .groupby(
            "roi_id",
            observed=True
        )
        .size()
    )


    count_df[
        desired
    ] = (
        count_df.index
        .to_series()
        .map(
            counts
        )
        .fillna(
            0
        )
        .astype(
            float
        )
        .to_numpy()
    )


count_df[
    "total_cells"
] = (
    count_df.index
    .to_series()
    .map(
        total_by_roi
    )
    .fillna(
        0
    )
    .astype(
        float
    )
    .to_numpy()
)


if (
    count_df[
        "total_cells"
    ]
    <= 0
).any():

    raise ValueError(
        "有 ROI 没有 cell metadata。"
    )


# ============================================================
# 9. Cell fractions
# ============================================================

fraction_df = (
    count_df[
        celltype_order
    ]
    .div(
        count_df[
            "total_cells"
        ],
        axis=0
    )
)


composition = (
    fraction_df.copy()
)


composition[
    "Disease"
] = (
    analysis_meta[
        "Disease"
    ].astype(str)
)


composition[
    "Patient"
] = (
    analysis_meta[
        "Patient_Sample_ID"
    ].astype(str)
)


composition[
    "PC1"
] = (
    analysis_meta[
        "PC1_crescent"
    ].astype(float)
)


# ============================================================
# 10. Formal composition models
# ============================================================

def fit_composition_models(
    data,
    outcome
):

    d = (
        data[
            [
                outcome,
                "Disease",
                "Patient",
                "PC1"
            ]
        ]
        .dropna()
        .copy()
    )


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),
        categories=[
            "ANCA",
            "SLE",
            "GBM"
        ]
    )


    # --------------------------------------------------------
    # patient-balanced weights
    # --------------------------------------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    # --------------------------------------------------------
    # Shared progression model
    # --------------------------------------------------------

    shared_fit = smf.wls(
        (
            f"{outcome} ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "+ C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    shared_terms = [
        term
        for term
        in shared_fit.params.index
        if (
            "bs(PC1"
            in term
            and
            ":"
            not in term
        )
    ]


    R_shared = np.zeros(
        (
            len(
                shared_terms
            ),
            len(
                shared_fit.params
            )
        )
    )


    for i, term in enumerate(
        shared_terms
    ):

        R_shared[
            i,
            shared_fit.params.index
            .get_loc(
                term
            )
        ] = 1


    shared_p = float(
        shared_fit.wald_test(
            R_shared,
            scalar=True
        ).pvalue
    )


    # --------------------------------------------------------
    # Disease × PC1 model
    # --------------------------------------------------------

    interaction_fit = smf.wls(
        (
            f"{outcome} ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    interaction_terms = [
        term
        for term
        in interaction_fit.params.index
        if (
            ":"
            in term
            and
            "C(Disease)"
            in term
            and
            "bs(PC1"
            in term
        )
    ]


    R_interaction = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                interaction_fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R_interaction[
            i,
            interaction_fit.params.index
            .get_loc(
                term
            )
        ] = 1


    interaction_p = float(
        interaction_fit.wald_test(
            R_interaction,
            scalar=True
        ).pvalue
    )


    return (
        d,
        shared_fit,
        interaction_fit,
        shared_p,
        interaction_p
    )


# ============================================================
# 11. Run all 8 cell populations
# ============================================================

result_rows = []

model_store = {}


for celltype in celltype_order:

    print(
        "Fitting:",
        celltype
    )


    (
        d,
        shared_fit,
        interaction_fit,
        shared_p,
        interaction_p
    ) = fit_composition_models(
        composition,
        celltype
    )


    model_store[
        celltype
    ] = {

        "data":
            d,

        "shared_fit":
            shared_fit,

        "interaction_fit":
            interaction_fit
    }


    result_rows.append({

        "celltype":
            celltype,

        "shared_PC1_pvalue":
            shared_p,

        "interaction_pvalue":
            interaction_p,

        "shared_rank":
            int(
                np.linalg.matrix_rank(
                    shared_fit.model.exog
                )
            ),

        "shared_columns":
            int(
                shared_fit.model.exog.shape[
                    1
                ]
            ),

        "interaction_rank":
            int(
                np.linalg.matrix_rank(
                    interaction_fit.model.exog
                )
            ),

        "interaction_columns":
            int(
                interaction_fit.model.exog.shape[
                    1
                ]
            )
    })


results = pd.DataFrame(
    result_rows
)


# ============================================================
# 12. BH-FDR
# ============================================================

results[
    "shared_PC1_FDR"
] = multipletests(
    results[
        "shared_PC1_pvalue"
    ],
    method="fdr_bh"
)[1]


results[
    "interaction_FDR"
] = multipletests(
    results[
        "interaction_pvalue"
    ],
    method="fdr_bh"
)[1]


results = (
    results
    .set_index(
        "celltype"
    )
    .loc[
        celltype_order
    ]
    .reset_index()
)


print(
    "\n================================"
)

print(
    "COMPOSITION MODEL RESULTS"
)

print(
    "================================"
)


display(
    results
)


# ============================================================
# 13. Prediction curves
# ============================================================

curve_celltypes = [
    "MAC",
    "FIB",
    "EC",
    "POD"
]


prediction_store = {}


for celltype in curve_celltypes:

    dat = (
        model_store[
            celltype
        ]
    )


    d = (
        dat[
            "data"
        ]
    )


    shared_fit = (
        dat[
            "shared_fit"
        ]
    )


    interaction_fit = (
        dat[
            "interaction_fit"
        ]
    )


    grid = np.linspace(
        d[
            "PC1"
        ].min(),
        d[
            "PC1"
        ].max(),
        180
    )


    disease_curves = {}


    for disease in [
        "ANCA",
        "SLE",
        "GBM"
    ]:

        newdata = pd.DataFrame({
            "PC1":
                grid,

            "Disease":
                pd.Categorical(
                    [disease]
                    *
                    len(grid),

                    categories=[
                        "ANCA",
                        "SLE",
                        "GBM"
                    ]
                )
        })


        disease_curves[
            disease
        ] = np.asarray(
            interaction_fit.predict(
                newdata
            )
        )


    shared_predictions = []


    for disease in [
        "ANCA",
        "SLE",
        "GBM"
    ]:

        newdata = pd.DataFrame({
            "PC1":
                grid,

            "Disease":
                pd.Categorical(
                    [disease]
                    *
                    len(grid),

                    categories=[
                        "ANCA",
                        "SLE",
                        "GBM"
                    ]
                )
        })


        shared_predictions.append(
            np.asarray(
                shared_fit.predict(
                    newdata
                )
            )
        )


    shared_curve = np.mean(
        np.vstack(
            shared_predictions
        ),
        axis=0
    )


    prediction_store[
        celltype
    ] = {

        "grid":
            grid,

        "disease_curves":
            disease_curves,

        "shared_curve":
            shared_curve
    }


# ============================================================
# 14. Figure style
# ============================================================

plt.rcParams[
    "pdf.fonttype"
] = 42

plt.rcParams[
    "ps.fonttype"
] = 42

plt.rcParams[
    "font.size"
] = 9.5

plt.rcParams[
    "axes.labelsize"
] = 10

plt.rcParams[
    "legend.fontsize"
] = 8


disease_colors = {

    "ANCA":
        "#56B4E9",

    "SLE":
        "#E69F00",

    "GBM":
        "#009E73"
}


disease_labels = {

    "ANCA":
        "ANCA",

    "SLE":
        "LN",

    "GBM":
        "anti-GBM"
}


# ============================================================
# 15. Compact FDR formatter
# ============================================================

def fmt_fdr(
    value
):

    value = float(
        value
    )


    if value < 0.001:

        return (
            f"{value:.1e}"
        )


    return (
        f"{value:.3f}"
    )


# ============================================================
# 16. Figure canvas
# ============================================================

fig = plt.figure(
    figsize=(
        17.5,
        8.6
    )
)


gs = fig.add_gridspec(
    2,
    4,
    height_ratios=[
        1.25,
        0.85
    ],
    hspace=0.42,
    wspace=0.42
)


axA = fig.add_subplot(
    gs[
        0,
        0
    ]
)

axB = fig.add_subplot(
    gs[
        0,
        1
    ]
)

axC = fig.add_subplot(
    gs[
        0,
        2
    ]
)

axD = fig.add_subplot(
    gs[
        0,
        3
    ]
)


axE = fig.add_subplot(
    gs[
        1,
        1:3
    ]
)


# ============================================================
# 17. Trajectory panel helper
# ============================================================

def draw_composition_panel(
    ax,
    celltype,
    panel
):

    data = (
        model_store[
            celltype
        ][
            "data"
        ]
    )


    curves = (
        prediction_store[
            celltype
        ]
    )


    for disease in [
        "ANCA",
        "SLE",
        "GBM"
    ]:

        raw = (
            data[
                data[
                    "Disease"
                ]
                .astype(str)
                ==
                disease
            ]
        )


        ax.scatter(
            raw[
                "PC1"
            ],
            raw[
                celltype
            ],
            s=13,
            alpha=0.18,
            color=disease_colors[
                disease
            ]
        )


        ax.plot(
            curves[
                "grid"
            ],
            curves[
                "disease_curves"
            ][
                disease
            ],
            linewidth=1.8,
            color=disease_colors[
                disease
            ],
            label=
                disease_labels[
                    disease
                ]
        )


    ax.plot(
        curves[
            "grid"
        ],
        curves[
            "shared_curve"
        ],
        linewidth=3.0,
        color="black",
        label="Shared trend"
    )


    result = (
        results[
            results[
                "celltype"
            ]
            ==
            celltype
        ]
        .iloc[
            0
        ]
    )


    ax.set_title(
        (
            f"{panel}  {celltype}\n"
            f"Shared PC1 FDR="
            f"{fmt_fdr(result['shared_PC1_FDR'])}; "
            f"Disease×PC1 FDR="
            f"{fmt_fdr(result['interaction_FDR'])}"
        ),
        loc="left",
        fontsize=8.8,
        fontweight="normal"
    )


    ax.set_xlabel(
        "Crescent-associated PC1"
    )


    ax.set_ylabel(
        "Cell fraction"
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ============================================================
# 18. Draw A–D
# ============================================================

draw_composition_panel(
    axA,
    "MAC",
    "A"
)


draw_composition_panel(
    axB,
    "FIB",
    "B"
)


draw_composition_panel(
    axC,
    "EC",
    "C"
)


draw_composition_panel(
    axD,
    "POD",
    "D"
)


axA.legend(
    frameon=False,
    fontsize=7.2,
    loc="best"
)


# ============================================================
# 19. PANEL E — Blue/white significance heatmap
# ============================================================

heatmap_values = np.column_stack([

    -np.log10(
        np.clip(
            results[
                "shared_PC1_FDR"
            ].to_numpy(),
            1e-20,
            1
        )
    ),

    -np.log10(
        np.clip(
            results[
                "interaction_FDR"
            ].to_numpy(),
            1e-20,
            1
        )
    )
])


# cap only for visualization
heatmap_display = np.clip(
    heatmap_values,
    0,
    20
)


im = axE.imshow(
    heatmap_display,
    aspect="auto",
    cmap="Blues",
    vmin=0,
    vmax=20
)


axE.set_xticks(
    [
        0,
        1
    ]
)


axE.set_xticklabels(
    [
        "Shared\nPC1 effect",
        "Disease × PC1\ninteraction"
    ]
)


axE.set_yticks(
    np.arange(
        len(
            celltype_order
        )
    )
)


axE.set_yticklabels(
    celltype_order
)


axE.set_title(
    (
        "E  Shared PC1 effects predominate "
        "across major cell populations"
    ),
    loc="left",
    fontweight="normal",
    fontsize=10.5
)


# ============================================================
# 20. FDR text annotations
# ============================================================

for i, row in results.iterrows():

    values = [
        row[
            "shared_PC1_FDR"
        ],
        row[
            "interaction_FDR"
        ]
    ]


    for j, value in enumerate(
        values
    ):

        display_value = (
            heatmap_display[
                i,
                j
            ]
        )


        text_color = (
            "white"
            if display_value > 9
            else "black"
        )


        axE.text(
            j,
            i,
            fmt_fdr(
                value
            ),
            ha="center",
            va="center",
            fontsize=8.2,
            color=text_color
        )


# ============================================================
# 21. Heatmap colorbar
# ============================================================

cbar = fig.colorbar(
    im,
    ax=axE,
    fraction=0.05,
    pad=0.04
)


cbar.set_label(
    "-log10(FDR)"
)


# ============================================================
# 22. Save final Figure 2
# ============================================================

png_path = (
    figdir /
    "Figure2_FINAL_shared_composition_progression_REVISED.png"
)


pdf_path = (
    figdir /
    "Figure2_FINAL_shared_composition_progression_REVISED.pdf"
)


plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 23. Save statistics
# ============================================================

stats_path = (
    base /
    "Figure2_FINAL_composition_statistics_REVISED.csv"
)


results.to_csv(
    stats_path,
    index=False
)


print(
    "\n========================================"
)

print(
    "FINAL REVISED FIGURE 2 COMPLETE"
)

print(
    "========================================"
)


print(
    "\nPNG:"
)

print(
    png_path
)


print(
    "\nPDF:"
)

print(
    pdf_path
)


print(
    "\nStatistics:"
)

print(
    stats_path
)


print(
    "\nPEC Disease×PC1 FDR =",
    float(
        results.loc[
            results[
                "celltype"
            ]
            ==
            "PEC",
            "interaction_FDR"
        ].iloc[
            0
        ]
    )
)

## Figure 3

In [ ]:
# ============================================================
# FINAL MANUSCRIPT FIGURE 3 — REVISED SUBMISSION VERSION
#
# Cell-type-resolved molecular trajectories diverge
# between LN and anti-GBM disease
#
# A  MAC molecular trajectory heatmap
# B  FIB molecular trajectory heatmap
# C  MAC innate/complement-associated module
# D  FIB wound-healing-associated module
# E  FIB complement-related module
# F  Panel-aware pathway enrichment
#
# IMPORTANT:
# - no biological results are recomputed/reselected
# - C/D/E show frozen-module slide sensitivity
# - A/B label only curated interpretable genes
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist


# ============================================================
# 1. Paths
# ============================================================

base = Path(
    str(PROJECT_DIR)
)

figdir = (
    base /
    "figures"
)

figdir.mkdir(
    parents=True,
    exist_ok=True
)


expr_path = (
    base /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)

module_path = (
    base /
    "figure5_MAC_FIB_trajectory_modules.csv"
)

score_path = (
    base /
    "figure5_frozen_module_scores.csv"
)

pathway_path = (
    base /
    "figure5_maintext_pathway_enrichment.csv"
)


# ============================================================
# 2. Check required files
# ============================================================

required_files = [
    expr_path,
    module_path,
    score_path,
    pathway_path
]


missing_files = [
    str(p)
    for p in required_files
    if not p.exists()
]


if len(
    missing_files
) > 0:

    raise FileNotFoundError(
        "缺少以下文件：\n"
        +
        "\n".join(
            missing_files
        )
    )


# ============================================================
# 3. Load saved analysis results
# ============================================================

expr_df = pd.read_csv(
    expr_path
)

module_df = pd.read_csv(
    module_path
)

module_scores = pd.read_csv(
    score_path
)

main_enrich = pd.read_csv(
    pathway_path
)


print(
    "Expression table:",
    expr_df.shape
)

print(
    "Frozen module scores:",
    module_scores.shape
)

print(
    "\nFrozen modules:"
)

print(
    module_scores[
        "module_label"
    ].value_counts()
)


# ============================================================
# 4. Figure typography
# ============================================================

plt.rcParams[
    "pdf.fonttype"
] = 42

plt.rcParams[
    "ps.fonttype"
] = 42

plt.rcParams[
    "font.size"
] = 9

plt.rcParams[
    "axes.titlesize"
] = 10.5

plt.rcParams[
    "axes.labelsize"
] = 9.5

plt.rcParams[
    "legend.fontsize"
] = 7.5

plt.rcParams[
    "xtick.labelsize"
] = 8.5

plt.rcParams[
    "ytick.labelsize"
] = 8.2


# ============================================================
# 5. Recreate normalized log-expression
#
# Same normalization used in Figure 5 discovery:
# mean counts → library size 10,000 → log1p
# ============================================================

metadata_cols = [
    "roi_id",
    "celltype_l1",
    "n_cells",
    "Disease",
    "Patient",
    "PC1"
]


gene_cols = [
    c
    for c in expr_df.columns
    if c not in metadata_cols
]


X_raw = (
    expr_df[
        gene_cols
    ]
    .to_numpy(
        dtype=float
    )
)


row_sum = (
    X_raw.sum(
        axis=1
    )
)


if (
    row_sum <= 0
).any():

    raise ValueError(
        "存在总表达量为 0 的 ROI×celltype unit。"
    )


X_norm = (
    X_raw
    /
    row_sum[:, None]
    *
    10000.0
)


X_log = np.log1p(
    X_norm
)


gene_index = {
    gene: i
    for i, gene
    in enumerate(
        gene_cols
    )
}


# ============================================================
# 6. Formal gene-level trajectory helper
#
# Used ONLY to reconstruct A/B heatmaps.
#
# Model:
# log-expression ~ spline(PC1) * Disease
#
# Patient-balanced
# Patient-clustered covariance
#
# Output:
# predicted anti-GBM minus LN trajectory
# ============================================================

def gene_delta_curve(
    celltype,
    gene,
    n_grid=100
):

    if gene not in gene_index:

        return None


    mask = (
        expr_df[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
        ==
        celltype
    )


    meta = (
        expr_df.loc[
            mask,
            metadata_cols
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    y = (
        X_log[
            mask,
            gene_index[
                gene
            ]
        ]
    )


    d = (
        meta[
            [
                "Disease",
                "Patient",
                "PC1"
            ]
        ]
        .copy()
    )


    d[
        "y"
    ] = y


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    # --------------------------------
    # patient-balanced weights
    # --------------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    # --------------------------------
    # formal model
    # --------------------------------

    fit = smf.wls(
        (
            "y ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    grid = np.linspace(
        d[
            "PC1"
        ].min(),
        d[
            "PC1"
        ].max(),
        n_grid
    )


    predictions = {}


    for disease in [
        "SLE",
        "GBM"
    ]:

        newdata = pd.DataFrame({
            "PC1":
                grid,

            "Disease":
                pd.Categorical(
                    [disease]
                    *
                    len(grid),

                    categories=[
                        "SLE",
                        "GBM"
                    ]
                )
        })


        predictions[
            disease
        ] = np.asarray(
            fit.predict(
                newdata
            )
        )


    delta = (
        predictions[
            "GBM"
        ]
        -
        predictions[
            "SLE"
        ]
    )


    return (
        grid,
        delta
    )


# ============================================================
# 7. Build MAC/FIB heatmap matrices
# ============================================================

heatmap_data = {}


for celltype in [
    "MAC",
    "FIB"
]:

    genes = (
        module_df.loc[
            module_df[
                "celltype"
            ]
            == celltype,
            "gene"
        ]
        .astype(str)
        .tolist()
    )


    curves = []

    kept_genes = []

    grid_saved = None


    print(
        "\nBuilding",
        celltype,
        "molecular trajectory heatmap..."
    )


    for gene in genes:

        result = gene_delta_curve(
            celltype,
            gene,
            n_grid=100
        )


        if result is None:

            continue


        grid, delta = result


        if not np.all(
            np.isfinite(
                delta
            )
        ):

            continue


        if (
            np.std(
                delta
            )
            < 1e-8
        ):

            continue


        # --------------------------------
        # Row-standardization
        #
        # Heatmap describes trajectory SHAPE,
        # not absolute magnitude.
        # --------------------------------

        z = (
            delta
            -
            np.mean(
                delta
            )
        ) / np.std(
            delta
        )


        curves.append(
            z
        )


        kept_genes.append(
            gene
        )


        grid_saved = grid


    zmat = np.vstack(
        curves
    )


    distance = pdist(
        zmat,
        metric="correlation"
    )


    Z = linkage(
        distance,
        method="average"
    )


    order = leaves_list(
        Z
    )


    heatmap_data[
        celltype
    ] = {
        "genes":
            np.array(
                kept_genes
            ),

        "zmat":
            zmat,

        "order":
            order,

        "grid":
            grid_saved
    }


# ============================================================
# 8. Curated heatmap labels
#
# IMPORTANT:
# Only labels are filtered.
# NO genes are removed from the heatmaps.
# ============================================================

preferred_labels = {

    "MAC": [
        "IL18",
        "GPR183",
        "S100A9",
        "PLAUR",
        "ITGAX",
        "CYBB",
        "RGS1"
    ],

    "FIB": [
        "ITGB1",
        "ITGA1",
        "CD44",
        "FAP",
        "COL16A1",
        "LUM",
        "C7"
    ]
}


# ============================================================
# 9. Frozen module model helper
#
# Used for C/D/E.
#
# Three specifications:
# 1. Unadjusted
# 2. Slide-adjusted
# 3. Overlap-slide restricted + slide adjustment
# ============================================================

def fit_module_model(
    data,
    add_slide=False
):

    d = data.dropna(
        subset=[
            "module_score",
            "Disease",
            "Patient",
            "PC1",
            "Slide"
        ]
    ).copy()


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    if add_slide:

        d[
            "Slide"
        ] = pd.Categorical(
            d[
                "Slide"
            ].astype(str)
        )


    # --------------------------------
    # patient-balanced
    # --------------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    if add_slide:

        formula = (
            "module_score ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease) "
            "+ C(Slide)"
        )

    else:

        formula = (
            "module_score ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        )


    fit = smf.wls(
        formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    # --------------------------------
    # Joint Disease × PC1 Wald test
    # --------------------------------

    interaction_terms = [
        term
        for term
        in fit.params.index
        if ":" in term
        and
        "C(Disease)"
        in term
        and
        "bs(PC1"
        in term
    ]


    R = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index
            .get_loc(
                term
            )
        ] = 1


    pvalue = float(
        fit.wald_test(
            R,
            scalar=True
        ).pvalue
    )


    return (
        d,
        fit,
        pvalue
    )


# ============================================================
# 10. Module difference-curve prediction
#
# anti-GBM minus LN
#
# For slide-adjusted models:
# predictions are averaged equally over slides.
# ============================================================

def predict_module_difference(
    fit,
    data,
    grid,
    add_slide=False
):

    # --------------------------------
    # Unadjusted model
    # --------------------------------

    if not add_slide:

        predictions = {}


        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({
                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(grid),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    )
            })


            predictions[
                disease
            ] = np.asarray(
                fit.predict(
                    newdata
                )
            )


        return (
            predictions[
                "GBM"
            ]
            -
            predictions[
                "SLE"
            ]
        )


    # --------------------------------
    # Slide-adjusted model
    # --------------------------------

    slide_categories = (
        data[
            "Slide"
        ].cat.categories
    )


    predictions = {
        "SLE": [],
        "GBM": []
    }


    for slide in slide_categories:

        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({
                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(grid),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    ),

                "Slide":
                    pd.Categorical(
                        [slide]
                        *
                        len(grid),

                        categories=
                            slide_categories
                    )
            })


            predictions[
                disease
            ].append(
                np.asarray(
                    fit.predict(
                        newdata
                    )
                )
            )


    mean_SLE = np.mean(
        np.vstack(
            predictions[
                "SLE"
            ]
        ),
        axis=0
    )


    mean_GBM = np.mean(
        np.vstack(
            predictions[
                "GBM"
            ]
        ),
        axis=0
    )


    return (
        mean_GBM
        -
        mean_SLE
    )


# ============================================================
# 11. Frozen biological modules
# ============================================================

frozen_modules = [

    {
        "label":
            "MAC M2",

        "title":
            "MAC innate/complement-associated module"
    },

    {
        "label":
            "FIB M1",

        "title":
            "FIB wound-healing-associated module"
    },

    {
        "label":
            "FIB M3",

        "title":
            "FIB complement-related module"
    }
]


# ============================================================
# 12. Fit the frozen modules
# ============================================================

module_plot_data = {}


for info in frozen_modules:

    label = (
        info[
            "label"
        ]
    )


    d0 = (
        module_scores[
            module_scores[
                "module_label"
            ]
            == label
        ]
        .copy()
    )


    # --------------------------------
    # Find disease-overlapping slides
    # --------------------------------

    slide_disease = (
        d0.groupby(
            "Slide",
            observed=True
        )[
            "Disease"
        ]
        .agg(
            lambda x:
            set(
                x.astype(str)
            )
        )
    )


    overlap_slides = [
        slide
        for slide, diseases
        in slide_disease.items()
        if (
            "SLE"
            in diseases
            and
            "GBM"
            in diseases
        )
    ]


    d_overlap = (
        d0[
            d0[
                "Slide"
            ]
            .astype(str)
            .isin(
                [
                    str(x)
                    for x in overlap_slides
                ]
            )
        ]
        .copy()
    )


    # --------------------------------
    # Model 1
    # --------------------------------

    (
        d_unadjusted,
        fit_unadjusted,
        p_unadjusted
    ) = fit_module_model(
        d0,
        add_slide=False
    )


    # --------------------------------
    # Model 2
    # --------------------------------

    (
        d_slide,
        fit_slide,
        p_slide
    ) = fit_module_model(
        d0,
        add_slide=True
    )


    # --------------------------------
    # Model 3
    # --------------------------------

    (
        d_overlap2,
        fit_overlap,
        p_overlap
    ) = fit_module_model(
        d_overlap,
        add_slide=True
    )


    # --------------------------------
    # Common PC1 prediction support
    # --------------------------------

    pc_low = max(
        d_unadjusted[
            "PC1"
        ].min(),

        d_slide[
            "PC1"
        ].min(),

        d_overlap2[
            "PC1"
        ].min()
    )


    pc_high = min(
        d_unadjusted[
            "PC1"
        ].max(),

        d_slide[
            "PC1"
        ].max(),

        d_overlap2[
            "PC1"
        ].max()
    )


    grid = np.linspace(
        pc_low,
        pc_high,
        150
    )


    # --------------------------------
    # Difference curves
    # --------------------------------

    curve_unadjusted = (
        predict_module_difference(
            fit_unadjusted,
            d_unadjusted,
            grid,
            add_slide=False
        )
    )


    curve_slide = (
        predict_module_difference(
            fit_slide,
            d_slide,
            grid,
            add_slide=True
        )
    )


    curve_overlap = (
        predict_module_difference(
            fit_overlap,
            d_overlap2,
            grid,
            add_slide=True
        )
    )


    module_plot_data[
        label
    ] = {

        "grid":
            grid,

        "Unadjusted":
            curve_unadjusted,

        "Slide-adjusted":
            curve_slide,

        "Overlap-slide restricted":
            curve_overlap,

        "p_unadjusted":
            p_unadjusted,

        "p_slide":
            p_slide,

        "p_overlap":
            p_overlap
    }


# ============================================================
# 13. Figure canvas
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(
        16,
        9.2
    ),
    constrained_layout=True
)


axA = axes[
    0,
    0
]

axB = axes[
    0,
    1
]

axC = axes[
    0,
    2
]

axD = axes[
    1,
    0
]

axE = axes[
    1,
    1
]

axF = axes[
    1,
    2
]


# ============================================================
# 14. Draw A/B heatmaps
# ============================================================

def draw_heatmap(
    ax,
    celltype,
    panel
):

    data = (
        heatmap_data[
            celltype
        ]
    )


    ordered = (
        data[
            "zmat"
        ][
            data[
                "order"
            ],
            :
        ]
    )


    ordered_genes = (
        data[
            "genes"
        ][
            data[
                "order"
            ]
        ]
    )


    im = ax.imshow(
        ordered,
        aspect="auto",
        interpolation="nearest",
        cmap="coolwarm",
        vmin=-2.5,
        vmax=2.5
    )


    grid = (
        data[
            "grid"
        ]
    )


    x_positions = np.linspace(
        0,
        len(grid) - 1,
        5
    )


    x_values = np.linspace(
        grid.min(),
        grid.max(),
        5
    )


    ax.set_xticks(
        x_positions
    )


    ax.set_xticklabels(
        [
            f"{x:.1f}"
            for x in x_values
        ]
    )


    # --------------------------------
    # curated labels only
    # --------------------------------

    wanted = set(
        preferred_labels[
            celltype
        ]
    )


    y_positions = []

    y_labels = []


    for position, gene in enumerate(
        ordered_genes
    ):

        if gene in wanted:

            y_positions.append(
                position
            )

            y_labels.append(
                gene
            )


    ax.set_yticks(
        y_positions
    )


    ax.set_yticklabels(
        y_labels,
        fontsize=7.2
    )


    ax.tick_params(
        axis="y",
        pad=2
    )


    ax.set_xlabel(
        "Crescent-associated PC1"
    )


    ax.set_ylabel(
        "Trajectory-divergent genes"
    )


    ax.set_title(
        (
            f"{panel}  "
            f"{celltype} molecular trajectories\n"
            "anti-GBM − LN"
        ),
        loc="left",
        fontweight="normal"
    )


    return im


imA = draw_heatmap(
    axA,
    "MAC",
    "A"
)


imB = draw_heatmap(
    axB,
    "FIB",
    "B"
)


# ============================================================
# 15. Draw C/D/E frozen module panels
# ============================================================

model_colors = {

    "Unadjusted":
        "#0072B2",

    "Slide-adjusted":
        "#E69F00",

    "Overlap-slide restricted":
        "#009E73"
}


def draw_module_panel(
    ax,
    module_label,
    title,
    panel
):

    dat = (
        module_plot_data[
            module_label
        ]
    )


    grid = (
        dat[
            "grid"
        ]
    )


    # --------------------------------
    # Unadjusted
    # --------------------------------

    ax.plot(
        grid,
        dat[
            "Unadjusted"
        ],
        linewidth=2.4,
        color=model_colors[
            "Unadjusted"
        ],
        label=(
            "Unadjusted "
            f"P={dat['p_unadjusted']:.2g}"
        )
    )


    # --------------------------------
    # Slide-adjusted
    # --------------------------------

    ax.plot(
        grid,
        dat[
            "Slide-adjusted"
        ],
        linewidth=2.2,
        color=model_colors[
            "Slide-adjusted"
        ],
        label=(
            "Slide-adjusted "
            f"P={dat['p_slide']:.2g}"
        )
    )


    # --------------------------------
    # Overlap-slide restricted
    # --------------------------------

    ax.plot(
        grid,
        dat[
            "Overlap-slide restricted"
        ],
        linewidth=2.2,
        color=model_colors[
            "Overlap-slide restricted"
        ],
        label=(
            "Overlap-slide restricted "
            f"P={dat['p_overlap']:.2g}"
        )
    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1,
        color="0.50"
    )


    ax.set_xlabel(
        "Crescent-associated PC1"
    )


    ax.set_ylabel(
        "Predicted anti-GBM − LN\nmodule score"
    )


    ax.set_title(
        f"{panel}  {title}",
        loc="left",
        fontweight="normal"
    )


    ax.legend(
        frameon=False,
        loc="best",
        fontsize=7.1,
        handlelength=2.3
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


draw_module_panel(
    axC,
    "MAC M2",
    "MAC innate/complement-associated module",
    "C"
)


draw_module_panel(
    axD,
    "FIB M1",
    "FIB wound-healing-associated module",
    "D"
)


draw_module_panel(
    axE,
    "FIB M3",
    "FIB complement-related module",
    "E"
)


# ============================================================
# 16. Prepare Panel F pathway data
# ============================================================

plot_enrich = (
    main_enrich.copy()
)


term_map = {

    "Wound Healing, Spreading of Cells (GO:0044319)":
        "Wound healing",

    "Terminal Pathway of Complement":
        "Terminal complement",

    "Innate Immune System":
        "Innate immune system",

    "Neutrophil Degranulation":
        "Neutrophil degranulation",

    "Complement Cascade":
        "Complement cascade",

    "Regulation of Complement Cascade":
        "Complement regulation"
}


plot_enrich[
    "display_term"
] = (
    plot_enrich[
        "term"
    ]
    .map(
        term_map
    )
    .fillna(
        plot_enrich[
            "term"
        ]
    )
)


plot_enrich[
    "module_label"
] = (
    plot_enrich[
        "celltype"
    ]
    +
    " M"
    +
    plot_enrich[
        "module"
    ].astype(str)
)


plot_enrich = (
    plot_enrich
    .sort_values(
        [
            "celltype",
            "module",
            "FDR_module_all_libraries"
        ],
        ascending=[
            True,
            True,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 17. Draw Panel F
# ============================================================

y = np.arange(
    len(
        plot_enrich
    )
)


x = (
    -np.log10(
        plot_enrich[
            "FDR_module_all_libraries"
        ]
    )
)


point_sizes = (
    plot_enrich[
        "overlap_n"
    ]
    *
    42
)


sc = axF.scatter(
    x,
    y,
    s=point_sizes,
    c=plot_enrich[
        "fold_enrichment"
    ],
    cmap="viridis",
    edgecolor="black",
    linewidth=0.55
)


axF.set_yticks(
    y
)


axF.set_yticklabels(
    [
        (
            f"{row.module_label}: "
            f"{row.display_term}"
        )
        for row
        in plot_enrich.itertuples()
    ],
    fontsize=7.3
)


axF.set_xlabel(
    "-log10(panel-aware FDR)"
)


axF.set_title(
    "F  Panel-aware pathway enrichment",
    loc="left",
    fontweight="normal"
)


axF.spines[
    "top"
].set_visible(
    False
)


axF.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# 18. Dot-size legend
#
# Upper-right area is intentionally used because
# the pathway points occupy mainly left/middle x-range.
# ============================================================

legend_sizes = [
    3,
    6,
    9
]


size_handles = []


for n in legend_sizes:

    size_handles.append(
        axF.scatter(
            [],
            [],
            s=n * 42,
            facecolor="white",
            edgecolor="black",
            linewidth=0.6,
            label=f"{n} genes"
        )
    )


size_legend = axF.legend(
    handles=size_handles,
    title="Overlap genes",
    frameon=False,
    loc="upper right",
    fontsize=7,
    title_fontsize=7.3
)


axF.add_artist(
    size_legend
)


# ============================================================
# 19. Heatmap colorbar
# ============================================================

heatmap_cbar = fig.colorbar(
    imA,
    ax=[
        axA,
        axB
    ],
    shrink=0.82,
    pad=0.02
)


heatmap_cbar.set_label(
    "Within-gene standardized\n"
    "anti-GBM − LN difference"
)


# ============================================================
# 20. Pathway colorbar
# ============================================================

pathway_cbar = fig.colorbar(
    sc,
    ax=axF,
    shrink=0.78,
    pad=0.03
)


pathway_cbar.set_label(
    "Fold enrichment"
)


# ============================================================
# 21. Save final manuscript Figure 3
# ============================================================

png_path = (
    figdir /
    "Figure3_FINAL_molecular_trajectory_divergence_REVISED.png"
)


pdf_path = (
    figdir /
    "Figure3_FINAL_molecular_trajectory_divergence_REVISED.pdf"
)


plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 22. Confirmation
# ============================================================

print(
    "\n========================================"
)

print(
    "FINAL REVISED FIGURE 3 COMPLETE"
)

print(
    "========================================"
)


print(
    "\nPNG:"
)

print(
    png_path
)


print(
    "\nPDF:"
)

print(
    pdf_path
)


print(
    "\nNo statistical results were re-selected."
)

print(
    "Only final manuscript visualization was revised."
)

## Figure 4

In [ ]:
# ============================================================
# FINAL MANUSCRIPT FIGURE 4 — REVISED SUBMISSION VERSION
#
# Macrophage–fibroblast neighborhood enrichment shows
# a disease-associated progression pattern with sensitivity
# to Xenium-slide adjustment
#
# A  Primary MAC–FIB neighborhood-enrichment trajectory
# B  Neighborhood-scale sensitivity (k = 4/6/8/10)
# C  Leave-one-anti-GBM-patient-out sensitivity
# D  Xenium-slide adjustment diagnostic
#
# IMPORTANT:
# - MAC–FIB refers to neighborhood enrichment / spatial association
# - no causal direction is implied
# - slide-adjusted negative sensitivity is shown transparently
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs


# ============================================================
# 1. Paths
# ============================================================

base = Path(
    str(PROJECT_DIR)
)

figdir = (
    base /
    "figures"
)

figdir.mkdir(
    parents=True,
    exist_ok=True
)


spatial_path = (
    base /
    "figure4_driver_neighbor_data_smoothed.csv"
)

formal_result_path = (
    base /
    "figure4_driver_results_smoothed.csv"
)

k_path = (
    base /
    "figure4_smoothed_k_sensitivity.csv"
)

loo_path = (
    base /
    "figure4_smoothed_GBM_LOO.csv"
)

roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)


# ============================================================
# 2. Check required files
# ============================================================

required_files = [
    spatial_path,
    formal_result_path,
    k_path,
    loo_path,
    roi_path
]


missing_files = [
    str(p)
    for p in required_files
    if not p.exists()
]


if len(
    missing_files
) > 0:

    raise FileNotFoundError(
        "缺少以下文件：\n"
        +
        "\n".join(
            missing_files
        )
    )


# ============================================================
# 3. Load saved results
# ============================================================

spatial = pd.read_csv(
    spatial_path,
    index_col=0
)

formal_results = pd.read_csv(
    formal_result_path
)

k_results = pd.read_csv(
    k_path
)

loo_results = pd.read_csv(
    loo_path
)

roi_ad = ad.read_h5ad(
    roi_path
)


spatial.index = (
    spatial.index.astype(str)
)

roi_meta = (
    roi_ad.obs.copy()
)

roi_meta.index = (
    roi_meta.index.astype(str)
)


print(
    "Spatial table:",
    spatial.shape
)


print(
    "\nFormal results:"
)

display(
    formal_results
)


# ============================================================
# 4. Reconstruct Xenium slide ID
# ============================================================

if "Biopsy_ID" in roi_meta.columns:

    roi_meta[
        "Slide"
    ] = (
        roi_meta[
            "Biopsy_ID"
        ]
        .astype(str)
        .str.extract(
            r"(\d{7})",
            expand=False
        )
    )

else:

    roi_meta[
        "Slide"
    ] = (
        roi_meta
        .index
        .to_series()
        .str.extract(
            r"(\d{7})",
            expand=False
        )
        .values
    )


# map slide onto spatial table
spatial[
    "Slide"
] = (
    spatial.index
    .to_series()
    .map(
        roi_meta[
            "Slide"
        ]
    )
    .values
)


if (
    spatial[
        "Slide"
    ].isna().any()
):

    print(
        "Warning: some spatial ROIs "
        "could not be assigned to a slide."
    )


# ============================================================
# 5. Keep LN vs anti-GBM primary analysis data
# ============================================================

d0 = (
    spatial[
        spatial[
            "Disease"
        ].isin(
            [
                "SLE",
                "GBM"
            ]
        )
    ]
    .dropna(
        subset=[
            "MAC_to_FIB",
            "Disease",
            "Patient",
            "PC1",
            "Slide"
        ]
    )
    .copy()
)


print(
    "\n================================"
)

print(
    "PRIMARY MAC–FIB DATA"
)

print(
    "================================"
)


print(
    "ROIs:",
    len(d0)
)


print(
    "Patients:",
    d0[
        "Patient"
    ].nunique()
)


print(
    "LN patients:",
    d0.loc[
        d0[
            "Disease"
        ]
        ==
        "SLE",
        "Patient"
    ].nunique()
)


print(
    "anti-GBM patients:",
    d0.loc[
        d0[
            "Disease"
        ]
        ==
        "GBM",
        "Patient"
    ].nunique()
)


# ============================================================
# 6. Formal spatial model helper
#
# patient-balanced
# patient-clustered covariance
# cubic B-spline df=3
#
# Optional:
# + C(Slide)
# ============================================================

def fit_spatial_model(
    data,
    add_slide=False
):

    d = data.dropna(
        subset=[
            "MAC_to_FIB",
            "Disease",
            "Patient",
            "PC1",
            "Slide"
        ]
    ).copy()


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    if add_slide:

        d[
            "Slide"
        ] = pd.Categorical(
            d[
                "Slide"
            ].astype(str)
        )


    # --------------------------------------------------------
    # Patient-balanced weights
    # Every patient contributes total weight ~1
    # --------------------------------------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    # --------------------------------------------------------
    # Formula
    # --------------------------------------------------------

    if add_slide:

        formula = (
            "MAC_to_FIB ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease) "
            "+ C(Slide)"
        )

    else:

        formula = (
            "MAC_to_FIB ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        )


    # --------------------------------------------------------
    # WLS + patient-clustered covariance
    # --------------------------------------------------------

    fit = smf.wls(
        formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    # --------------------------------------------------------
    # Joint Disease × PC1 Wald test
    # --------------------------------------------------------

    interaction_terms = [
        term
        for term
        in fit.params.index
        if ":" in term
        and
        "C(Disease)"
        in term
        and
        "bs(PC1"
        in term
    ]


    if len(
        interaction_terms
    ) == 0:

        raise ValueError(
            "没有找到 Disease × PC1 interaction terms。"
        )


    R = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index
            .get_loc(
                term
            )
        ] = 1


    pvalue = float(
        fit.wald_test(
            R,
            scalar=True
        ).pvalue
    )


    return (
        d,
        fit,
        pvalue
    )


# ============================================================
# 7. Primary unadjusted model
# ============================================================

(
    d_primary,
    fit_primary,
    primary_p
) = fit_spatial_model(
    d0,
    add_slide=False
)


# ============================================================
# 8. Primary FDR from saved formal analysis
# ============================================================

primary_row = (
    formal_results[
        formal_results[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .copy()
)


if len(
    primary_row
) == 0:

    raise ValueError(
        "figure4_driver_results_smoothed.csv 中"
        "没有找到 MAC_to_FIB。"
    )


primary_fdr = float(
    primary_row[
        "FDR"
    ].iloc[0]
)


print(
    "\n================================"
)

print(
    "PRIMARY FORMAL RESULT"
)

print(
    "================================"
)


print(
    "Interaction P =",
    primary_p
)


print(
    "Interaction FDR =",
    primary_fdr
)


# ============================================================
# 9. Primary prediction curves + 95% CI
# ============================================================

primary_grid = np.linspace(
    d_primary[
        "PC1"
    ].min(),
    d_primary[
        "PC1"
    ].max(),
    180
)


primary_predictions = {}


for disease in [
    "SLE",
    "GBM"
]:

    newdata = pd.DataFrame({
        "PC1":
            primary_grid,

        "Disease":
            pd.Categorical(
                [disease]
                *
                len(
                    primary_grid
                ),

                categories=[
                    "SLE",
                    "GBM"
                ]
            )
    })


    prediction = (
        fit_primary
        .get_prediction(
            newdata
        )
        .summary_frame(
            alpha=0.05
        )
    )


    primary_predictions[
        disease
    ] = pd.DataFrame({
        "PC1":
            primary_grid,

        "mean":
            prediction[
                "mean"
            ].to_numpy(),

        "lower":
            prediction[
                "mean_ci_lower"
            ].to_numpy(),

        "upper":
            prediction[
                "mean_ci_upper"
            ].to_numpy()
    })


# ============================================================
# 10. Identify disease-overlapping Xenium slides
# ============================================================

slide_disease = (
    d0.groupby(
        "Slide",
        observed=True
    )[
        "Disease"
    ]
    .agg(
        lambda x:
        set(
            x.astype(str)
        )
    )
)


overlap_slides = [
    slide
    for slide, diseases
    in slide_disease.items()
    if (
        "SLE"
        in diseases
        and
        "GBM"
        in diseases
    )
]


print(
    "\nDisease-overlapping slides:"
)

print(
    overlap_slides
)


print(
    "Number of overlap slides:",
    len(
        overlap_slides
    )
)


d_overlap = (
    d0[
        d0[
            "Slide"
        ]
        .astype(str)
        .isin(
            [
                str(x)
                for x in overlap_slides
            ]
        )
    ]
    .copy()
)


# ============================================================
# 11. Slide sensitivity models
# ============================================================

(
    d_unadjusted,
    fit_unadjusted,
    p_unadjusted
) = fit_spatial_model(
    d0,
    add_slide=False
)


(
    d_slide,
    fit_slide,
    p_slide
) = fit_spatial_model(
    d0,
    add_slide=True
)


(
    d_overlap2,
    fit_overlap,
    p_overlap
) = fit_spatial_model(
    d_overlap,
    add_slide=True
)


# ============================================================
# 12. Common PC1 support for slide diagnostic
# ============================================================

diag_low = max(
    d_unadjusted[
        "PC1"
    ].min(),

    d_slide[
        "PC1"
    ].min(),

    d_overlap2[
        "PC1"
    ].min()
)


diag_high = min(
    d_unadjusted[
        "PC1"
    ].max(),

    d_slide[
        "PC1"
    ].max(),

    d_overlap2[
        "PC1"
    ].max()
)


diag_grid = np.linspace(
    diag_low,
    diag_high,
    150
)


# ============================================================
# 13. Disease-difference prediction helper
#
# anti-GBM minus LN
#
# Slide-adjusted models:
# equal-average over slides present in that model
# ============================================================

def predict_disease_difference(
    fit,
    data,
    grid,
    add_slide=False
):

    # --------------------------------------------------------
    # Unadjusted
    # --------------------------------------------------------

    if not add_slide:

        predictions = {}


        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({
                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(grid),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    )
            })


            predictions[
                disease
            ] = np.asarray(
                fit.predict(
                    newdata
                )
            )


        return (
            predictions[
                "GBM"
            ]
            -
            predictions[
                "SLE"
            ]
        )


    # --------------------------------------------------------
    # Slide-adjusted
    # --------------------------------------------------------

    slide_categories = (
        data[
            "Slide"
        ].cat.categories
    )


    predictions = {
        "SLE": [],
        "GBM": []
    }


    for slide in slide_categories:

        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({
                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(grid),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    ),

                "Slide":
                    pd.Categorical(
                        [slide]
                        *
                        len(grid),

                        categories=
                            slide_categories
                    )
            })


            predictions[
                disease
            ].append(
                np.asarray(
                    fit.predict(
                        newdata
                    )
                )
            )


    mean_SLE = np.mean(
        np.vstack(
            predictions[
                "SLE"
            ]
        ),
        axis=0
    )


    mean_GBM = np.mean(
        np.vstack(
            predictions[
                "GBM"
            ]
        ),
        axis=0
    )


    return (
        mean_GBM
        -
        mean_SLE
    )


# ============================================================
# 14. Slide diagnostic curves
# ============================================================

curve_unadjusted = (
    predict_disease_difference(
        fit_unadjusted,
        d_unadjusted,
        diag_grid,
        add_slide=False
    )
)


curve_slide = (
    predict_disease_difference(
        fit_slide,
        d_slide,
        diag_grid,
        add_slide=True
    )
)


curve_overlap = (
    predict_disease_difference(
        fit_overlap,
        d_overlap2,
        diag_grid,
        add_slide=True
    )
)


# ============================================================
# 15. Curve similarity
# ============================================================

corr_slide = float(
    np.corrcoef(
        curve_unadjusted,
        curve_slide
    )[
        0,
        1
    ]
)


corr_overlap = float(
    np.corrcoef(
        curve_unadjusted,
        curve_overlap
    )[
        0,
        1
    ]
)


amp_unadjusted = float(
    np.max(
        curve_unadjusted
    )
    -
    np.min(
        curve_unadjusted
    )
)


amp_slide = float(
    np.max(
        curve_slide
    )
    -
    np.min(
        curve_slide
    )
)


amp_overlap = float(
    np.max(
        curve_overlap
    )
    -
    np.min(
        curve_overlap
    )
)


print(
    "\n================================"
)

print(
    "SLIDE SENSITIVITY"
)

print(
    "================================"
)


print(
    "Unadjusted P =",
    p_unadjusted
)


print(
    "Slide-adjusted P =",
    p_slide
)


print(
    "Overlap-slide restricted P =",
    p_overlap
)


print(
    "\nCurve correlations:"
)


print(
    "Slide-adjusted vs unadjusted =",
    corr_slide
)


print(
    "Overlap-slide restricted vs unadjusted =",
    corr_overlap
)


# ============================================================
# 16. Prepare k sensitivity
# ============================================================

k_macfib = (
    k_results[
        k_results[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .copy()
    .sort_values(
        "k"
    )
)


if len(
    k_macfib
) == 0:

    raise ValueError(
        "k sensitivity file 中"
        "没有找到 MAC_to_FIB。"
    )


print(
    "\n================================"
)

print(
    "K SENSITIVITY"
)

print(
    "================================"
)


display(
    k_macfib
)


# ============================================================
# 17. Prepare leave-one-anti-GBM-out
# ============================================================

loo_macfib = (
    loo_results[
        loo_results[
            "relationship"
        ]
        ==
        "MAC_to_FIB"
    ]
    .copy()
)


if len(
    loo_macfib
) == 0:

    raise ValueError(
        "LOO file 中没有找到 MAC_to_FIB。"
    )


print(
    "\n================================"
)

print(
    "ANTI-GBM LEAVE-ONE-OUT"
)

print(
    "================================"
)


display(
    loo_macfib
)


# ============================================================
# 18. Figure typography
# ============================================================

plt.rcParams[
    "pdf.fonttype"
] = 42

plt.rcParams[
    "ps.fonttype"
] = 42

plt.rcParams[
    "font.size"
] = 9.5

plt.rcParams[
    "axes.titlesize"
] = 11

plt.rcParams[
    "axes.labelsize"
] = 10

plt.rcParams[
    "legend.fontsize"
] = 8

plt.rcParams[
    "xtick.labelsize"
] = 8.5

plt.rcParams[
    "ytick.labelsize"
] = 8.5


# ============================================================
# 19. Colors
# ============================================================

disease_colors = {

    "SLE":
        "#E69F00",

    "GBM":
        "#009E73"
}


disease_labels = {

    "SLE":
        "LN",

    "GBM":
        "anti-GBM"
}


model_colors = {

    "Unadjusted":
        "#0072B2",

    "Slide-adjusted":
        "#E69F00",

    "Overlap-slide restricted":
        "#009E73"
}


# ============================================================
# 20. Figure canvas
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        13.5,
        10
    ),
    constrained_layout=True
)


axA = axes[
    0,
    0
]

axB = axes[
    0,
    1
]

axC = axes[
    1,
    0
]

axD = axes[
    1,
    1
]


# ============================================================
# 21. PANEL A
# Primary MAC–FIB neighborhood enrichment trajectory
# ============================================================

for disease in [
    "SLE",
    "GBM"
]:

    raw = (
        d_primary[
            d_primary[
                "Disease"
            ]
            .astype(str)
            ==
            disease
        ]
    )


    prediction = (
        primary_predictions[
            disease
        ]
    )


    color = (
        disease_colors[
            disease
        ]
    )


    # raw ROI values
    axA.scatter(
        raw[
            "PC1"
        ],
        raw[
            "MAC_to_FIB"
        ],
        s=18,
        alpha=0.18,
        color=color
    )


    # formal fitted trajectory
    axA.plot(
        prediction[
            "PC1"
        ],
        prediction[
            "mean"
        ],
        linewidth=2.6,
        color=color,
        label=
            disease_labels[
                disease
            ]
    )


    # 95% CI
    axA.fill_between(
        prediction[
            "PC1"
        ],
        prediction[
            "lower"
        ],
        prediction[
            "upper"
        ],
        color=color,
        alpha=0.13
    )


axA.axhline(
    0,
    linestyle="--",
    linewidth=1,
    color="0.45"
)


axA.set_xlabel(
    "Crescent-associated PC1"
)


axA.set_ylabel(
    "log2 neighborhood enrichment"
)


axA.set_title(
    (
        "A  MAC–FIB neighborhood enrichment\n"
        "LN vs anti-GBM interaction "
        f"FDR = {primary_fdr:.3f}"
    ),
    loc="left",
    fontweight="normal"
)


axA.legend(
    frameon=False,
    loc="upper right"
)


axA.spines[
    "top"
].set_visible(
    False
)


axA.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# 22. PANEL B
# Neighborhood-scale sensitivity
# ============================================================

axB.plot(
    k_macfib[
        "k"
    ],
    k_macfib[
        "pvalue"
    ],
    marker="o",
    markersize=6,
    linewidth=2.2,
    color="#0072B2"
)


# nominal significance threshold
axB.axhline(
    0.05,
    linestyle="--",
    linewidth=1.2,
    color="0.45"
)


axB.set_yscale(
    "log"
)


axB.set_xticks(
    [
        4,
        6,
        8,
        10
    ]
)


axB.set_xlabel(
    "Nearest neighbors (k)"
)


axB.set_ylabel(
    "Disease × PC1 interaction P value"
)


axB.set_title(
    "B  Neighborhood-scale sensitivity",
    loc="left",
    fontweight="normal"
)


# threshold label
axB.text(
    0.02,
    0.97,
    "P = 0.05",
    transform=axB.transAxes,
    ha="left",
    va="top",
    fontsize=7.5,
    color="0.40"
)


# summary
axB.text(
    0.96,
    0.08,
    "4/4 nominal P < 0.05",
    transform=axB.transAxes,
    ha="right",
    va="bottom",
    fontsize=9
)


axB.spines[
    "top"
].set_visible(
    False
)


axB.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# 23. PANEL C
# Leave-one-anti-GBM-patient-out
# ============================================================

x_positions = np.arange(
    len(
        loo_macfib
    )
)


axC.plot(
    x_positions,
    loo_macfib[
        "pvalue"
    ],
    marker="o",
    markersize=6,
    linewidth=2.2,
    color="#0072B2"
)


axC.axhline(
    0.05,
    linestyle="--",
    linewidth=1.2,
    color="0.45"
)


axC.set_yscale(
    "log"
)


axC.set_xticks(
    x_positions
)


axC.set_xticklabels(
    loo_macfib[
        "dropped_GBM_patient"
    ],
    rotation=45,
    ha="right"
)


axC.set_xlabel(
    "anti-GBM patient excluded"
)


axC.set_ylabel(
    "Disease × PC1 interaction P value"
)


axC.set_title(
    "C  Leave-one-anti-GBM-patient-out",
    loc="left",
    fontweight="normal"
)


# threshold label
axC.text(
    0.02,
    0.97,
    "P = 0.05",
    transform=axC.transAxes,
    ha="left",
    va="top",
    fontsize=7.5,
    color="0.40"
)


# sensitivity summary
axC.text(
    0.96,
    0.08,
    "5/5 nominal P < 0.05",
    transform=axC.transAxes,
    ha="right",
    va="bottom",
    fontsize=9
)


axC.spines[
    "top"
].set_visible(
    False
)


axC.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# 24. PANEL D
# Xenium-slide sensitivity
# ============================================================

axD.plot(
    diag_grid,
    curve_unadjusted,
    linewidth=2.5,
    color=model_colors[
        "Unadjusted"
    ],
    label=(
        "Unadjusted "
        f"P={p_unadjusted:.3g}"
    )
)


axD.plot(
    diag_grid,
    curve_slide,
    linewidth=2.3,
    color=model_colors[
        "Slide-adjusted"
    ],
    label=(
        "Slide-adjusted "
        f"P={p_slide:.3g}"
    )
)


axD.plot(
    diag_grid,
    curve_overlap,
    linewidth=2.3,
    color=model_colors[
        "Overlap-slide restricted"
    ],
    label=(
        "Overlap-slide restricted "
        f"P={p_overlap:.3g}"
    )
)


axD.axhline(
    0,
    linestyle="--",
    linewidth=1,
    color="0.45"
)


axD.set_xlabel(
    "Crescent-associated PC1"
)


axD.set_ylabel(
    "Predicted anti-GBM − LN\n"
    "MAC–FIB neighborhood enrichment"
)


axD.set_title(
    "D  Xenium-slide sensitivity",
    loc="left",
    fontweight="normal"
)


axD.legend(
    frameon=False,
    loc="lower right",
    fontsize=7.5,
    handlelength=2.3
)


# ------------------------------------------------------------
# Effect-geometry annotation
# moved inward to avoid touching frame
# ------------------------------------------------------------

axD.text(
    0.06,
    0.10,
    (
        "Curve correlation vs unadjusted:\n"
        f"slide-adjusted r = "
        f"{corr_slide:.2f}\n"
        "overlap-slide restricted r = "
        f"{corr_overlap:.2f}"
    ),
    transform=axD.transAxes,
    ha="left",
    va="bottom",
    fontsize=7.8
)


axD.spines[
    "top"
].set_visible(
    False
)


axD.spines[
    "right"
].set_visible(
    False
)


# ============================================================
# 25. Save final submission Figure 4
# ============================================================

png_path = (
    figdir /
    "Figure4_FINAL_MAC_FIB_neighborhood_enrichment_REVISED.png"
)


pdf_path = (
    figdir /
    "Figure4_FINAL_MAC_FIB_neighborhood_enrichment_REVISED.pdf"
)


plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 26. Save a compact final statistical summary
# ============================================================

final_summary = pd.DataFrame({

    "analysis": [
        "Primary unadjusted",
        "Slide-adjusted",
        "Overlap-slide restricted"
    ],

    "interaction_p": [
        p_unadjusted,
        p_slide,
        p_overlap
    ],

    "curve_correlation_vs_unadjusted": [
        1.0,
        corr_slide,
        corr_overlap
    ],

    "curve_amplitude": [
        amp_unadjusted,
        amp_slide,
        amp_overlap
    ],

    "n_ROI": [
        len(
            d_unadjusted
        ),
        len(
            d_slide
        ),
        len(
            d_overlap2
        )
    ],

    "n_patients": [
        d_unadjusted[
            "Patient"
        ].nunique(),

        d_slide[
            "Patient"
        ].nunique(),

        d_overlap2[
            "Patient"
        ].nunique()
    ],

    "n_GBM_patients": [
        d_unadjusted.loc[
            d_unadjusted[
                "Disease"
            ]
            .astype(str)
            ==
            "GBM",
            "Patient"
        ].nunique(),

        d_slide.loc[
            d_slide[
                "Disease"
            ]
            .astype(str)
            ==
            "GBM",
            "Patient"
        ].nunique(),

        d_overlap2.loc[
            d_overlap2[
                "Disease"
            ]
            .astype(str)
            ==
            "GBM",
            "Patient"
        ].nunique()
    ]
})


summary_path = (
    base /
    "Figure4_FINAL_statistical_summary.csv"
)


final_summary.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 27. Final confirmation
# ============================================================

print(
    "\n========================================"
)

print(
    "FINAL REVISED FIGURE 4 COMPLETE"
)

print(
    "========================================"
)


print(
    "\nPrimary FDR:"
)

print(
    primary_fdr
)


print(
    "\nPrimary unadjusted P:"
)

print(
    p_unadjusted
)


print(
    "\nSlide-adjusted P:"
)

print(
    p_slide
)


print(
    "\nOverlap-slide restricted P:"
)

print(
    p_overlap
)


print(
    "\nCurve correlations:"
)

print(
    "Slide-adjusted vs unadjusted =",
    corr_slide
)


print(
    "Overlap-slide restricted vs unadjusted =",
    corr_overlap
)


print(
    "\nPNG:"
)

print(
    png_path
)


print(
    "\nPDF:"
)

print(
    pdf_path
)


print(
    "\nStatistical summary:"
)

print(
    summary_path
)


print(
    "\nNo statistical results were re-selected."
)

print(
    "Only final manuscript visualization was revised."
)